# 01 — Bronze Ingest: Project Profitability & Utilisation Analytics
Reads the four raw CSV files from the Lakehouse `Files/raw/` area and writes them as Delta tables in the Bronze schema.

| Table | Source |
|---|---|
| bronze_consultants | consultants.csv |
| bronze_clients | clients.csv |
| bronze_engagements | engagements.csv |
| bronze_timesheets | timesheets.csv |

In [ ]:
LAKEHOUSE_NAME = "proservices_lakehouse"
RAW_PATH       = "Files/raw"
BRONZE_SCHEMA  = "bronze"

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType
)

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {LAKEHOUSE_NAME}.{BRONZE_SCHEMA}")
notebookutils.fs.mkdirs("Files/raw")
print(f"Schema '{BRONZE_SCHEMA}' ready.")

In [ ]:
# ── consultants ──────────────────────────────────────────────────────────────
consultants_schema = StructType([
    StructField("consultant_id",    StringType(),  False),
    StructField("grade",            StringType(),  True),
    StructField("practice",         StringType(),  True),
    StructField("region",           StringType(),  True),
    StructField("daily_rate_gbp",   IntegerType(), True),
    StructField("years_experience", IntegerType(), True),
    StructField("is_billable",      IntegerType(), True),
    StructField("hire_date",        StringType(),  True),
])

df_con = (
    spark.read.option("header", True).schema(consultants_schema)
    .csv(f"{RAW_PATH}/consultants.csv")
    .withColumn("hire_date",    F.to_date("hire_date"))
    .withColumn("_ingested_at", F.current_timestamp())
)

df_con.write.format("delta").mode("overwrite").option("overwriteSchema", True) \
    .saveAsTable(f"{LAKEHOUSE_NAME}.{BRONZE_SCHEMA}.bronze_consultants")
print(f"bronze_consultants: {df_con.count():,}")

In [ ]:
# ── clients ──────────────────────────────────────────────────────────────────
clients_schema = StructType([
    StructField("client_id",            StringType(), False),
    StructField("client_name",          StringType(), True),
    StructField("industry",             StringType(), True),
    StructField("region",               StringType(), True),
    StructField("tier",                 StringType(), True),
    StructField("contract_value_gbp",   DoubleType(), True),
    StructField("relationship_years",   IntegerType(),True),
    StructField("nps_score",            IntegerType(),True),
])

df_cli = (
    spark.read.option("header", True).schema(clients_schema)
    .csv(f"{RAW_PATH}/clients.csv")
    .withColumn("_ingested_at", F.current_timestamp())
)

df_cli.write.format("delta").mode("overwrite").option("overwriteSchema", True) \
    .saveAsTable(f"{LAKEHOUSE_NAME}.{BRONZE_SCHEMA}.bronze_clients")
print(f"bronze_clients: {df_cli.count():,}")

In [ ]:
# ── engagements ──────────────────────────────────────────────────────────────
engagements_schema = StructType([
    StructField("engagement_id",       StringType(), False),
    StructField("client_id",           StringType(), True),
    StructField("lead_consultant_id",  StringType(), True),
    StructField("practice",            StringType(), True),
    StructField("start_date",          StringType(), True),
    StructField("planned_end_date",    StringType(), True),
    StructField("budget_gbp",          DoubleType(), True),
    StructField("actual_spend_gbp",    DoubleType(), True),
    StructField("margin_pct",          DoubleType(), True),
    StructField("status",              StringType(), True),
    StructField("headcount",           IntegerType(),True),
])

df_eng = (
    spark.read.option("header", True).schema(engagements_schema)
    .csv(f"{RAW_PATH}/engagements.csv")
    .withColumn("start_date",       F.to_date("start_date"))
    .withColumn("planned_end_date", F.to_date("planned_end_date"))
    .withColumn("_ingested_at",     F.current_timestamp())
)

df_eng.write.format("delta").mode("overwrite").option("overwriteSchema", True) \
    .saveAsTable(f"{LAKEHOUSE_NAME}.{BRONZE_SCHEMA}.bronze_engagements")
print(f"bronze_engagements: {df_eng.count():,}")

In [ ]:
# ── timesheets ───────────────────────────────────────────────────────────────
timesheets_schema = StructType([
    StructField("timesheet_id",    StringType(), False),
    StructField("consultant_id",   StringType(), True),
    StructField("engagement_id",   StringType(), True),
    StructField("week_starting",   StringType(), True),
    StructField("task_type",       StringType(), True),
    StructField("hours_logged",    DoubleType(), True),
    StructField("is_billable",     IntegerType(),True),
    StructField("daily_rate_gbp",  IntegerType(),True),
    StructField("billed_value_gbp",DoubleType(), True),
])

df_ts = (
    spark.read.option("header", True).schema(timesheets_schema)
    .csv(f"{RAW_PATH}/timesheets.csv")
    .withColumn("week_starting", F.to_date("week_starting"))
    .withColumn("_ingested_at",  F.current_timestamp())
)

df_ts.write.format("delta").mode("overwrite").option("overwriteSchema", True) \
    .saveAsTable(f"{LAKEHOUSE_NAME}.{BRONZE_SCHEMA}.bronze_timesheets")
print(f"bronze_timesheets: {df_ts.count():,}")

In [ ]:
print("\n=== Bronze Ingest Complete ===")
for tbl in ["bronze_consultants", "bronze_clients", "bronze_engagements", "bronze_timesheets"]:
    cnt = spark.table(f"{LAKEHOUSE_NAME}.{BRONZE_SCHEMA}.{tbl}").count()
    print(f"  {tbl}: {cnt:,}")